# Notebook 07: Synthetic Phase Recovery + Model Comparison

**Goal:**
1. Compare Standard QM vs phenomenological detector-shift model vs physical X-Theta tensor model.
2. Demonstrate the recovery of small angular phases using direct curve fitting.
3. Show why CHSH-magnitude-only recovery is weak near zero.
4. Demonstrate the X-Theta residual signature test.

> **Scientific Scope:** The detector-shift parameter `delta` and the physical X-Theta parameter `Phi` are distinct. `delta` tests phase-recovery methodology; `Phi` tests the tensor-anisotropy model used in the paper.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

from scipy.optimize import curve_fit
from scipy.stats import chi2

# Ensure output directory exists
output_dir = Path("../outputs/synthetic_recovery")
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Theory Models

We distinguish between three models:
1. **Standard QM:** isotropic singlet correlation.
2. **Detector-Shift (Model A):** phenomenological phase shift $\delta$.
3. **X-Theta Tensor (Model B):** anisotropic tensor deformation controlled by $\Phi$.

In [ ]:
def qm_standard(theta_A, theta_B):
    """
    Standard flat-space singlet correlation: E = -cos(theta_A - theta_B)
    """
    return -np.cos(theta_A - theta_B)


def model_a_detector_shift(theta_A, theta_B, delta):
    """
    Model A: Phenomenological detector phase shift.
    E = -cos(theta_A - theta_B + delta)
    """
    return -np.cos(theta_A - theta_B + delta)


def model_b_xtheta_tensor(theta_A, theta_B, Phi):
    """
    Model B: Physical X-Theta tensor anisotropy.
    E = a^T T(Phi) b
    T(Phi) = diag[-cos(2*Phi), -cos(2*Phi), -1]
    For equatorial measurements (z=0):
    E = -cos(2*Phi) * cos(theta_A - theta_B)
    """
    # For simplicity in this notebook, we assume measurements are in the XY plane.
    # a = [cos(theta_A), sin(theta_A), 0]
    # b = [cos(theta_B), sin(theta_B), 0]
    # E = -cos(2*Phi) * (cos(theta_A)cos(theta_B) + sin(theta_A)sin(theta_B))
    return -np.cos(2 * Phi) * np.cos(theta_A - theta_B)


def chsh_s_from_delta(delta):
    """
    For ideal optimal CHSH settings with common phase shift delta:
    |S(delta)| = 2 sqrt(2) |cos(delta)|
    """
    return 2 * np.sqrt(2) * abs(np.cos(delta))


def delta_from_chsh_s(S):
    """
    Invert ideal CHSH formula for delta.
    """
    ratio = np.clip(abs(S) / (2 * np.sqrt(2)), 0.0, 1.0)
    return np.arccos(ratio)

def s_max_from_phi(Phi):
    """
    Physical S_max for X-Theta tensor model (Horodecki maximum).
    S_max(Phi) = 2 sqrt(1 + cos^2(2*Phi))
    """
    return 2 * np.sqrt(1 + np.cos(2 * Phi)**2)

## 2. Synthetic Data Generation

In [ ]:
def make_chsh4_settings():
    return pd.DataFrame([
        {"setting": "00", "theta_A": 0.0,       "theta_B": np.pi / 4},
        {"setting": "01", "theta_A": 0.0,       "theta_B": -np.pi / 4},
        {"setting": "10", "theta_A": np.pi / 2, "theta_B": np.pi / 4},
        {"setting": "11", "theta_A": np.pi / 2, "theta_B": -np.pi / 4},
    ])


def make_angle_scan_settings(n_angles=41):
    angle_diffs = np.linspace(-np.pi, np.pi, n_angles)
    return pd.DataFrame({
        "setting": [f"scan_{i:02d}" for i in range(n_angles)],
        "theta_A": angle_diffs,
        "theta_B": np.zeros_like(angle_diffs),
    })


def simulate_data(
    model_func,
    param_val,
    settings_df,
    n_trials_per_setting=10000,
    seed=42
):
    rng = np.random.default_rng(seed)
    rows = []

    for row in settings_df.itertuples(index=False):
        E_true = model_func(row.theta_A, row.theta_B, param_val)
        p_plus = np.clip((1.0 + E_true) / 2.0, 0.0, 1.0)
        
        products = rng.choice([+1, -1], size=n_trials_per_setting, p=[p_plus, 1.0 - p_plus])
        E_hat = products.mean()
        error = np.sqrt(max(1.0 - E_hat**2, 1e-12) / n_trials_per_setting)

        rows.append({
            "setting": row.setting,
            "theta_A": row.theta_A,
            "theta_B": row.theta_B,
            "angle_diff": row.theta_A - row.theta_B,
            "E": E_hat,
            "Error": max(error, 1e-9)
        })

    return pd.DataFrame(rows)

## 3. Recovery and Comparison Logic

In [ ]:
def weighted_chi2(y, y_pred, errors):
    return float(np.sum(((y - y_pred) / errors) ** 2))


def run_comparison(df, true_model_type, true_param):
    x_A = df["theta_A"].values
    x_B = df["theta_B"].values
    y = df["E"].values
    err = df["Error"].values

    # Fit Model A (delta)
    popt_a, _ = curve_fit(model_a_detector_shift, (x_A, x_B), y, sigma=err, p0=[0.0])
    delta_hat = popt_a[0]
    chi2_a = weighted_chi2(y, model_a_detector_shift(x_A, x_B, delta_hat), err)

    # Fit Model B (Phi)
    popt_b, _ = curve_fit(model_b_xtheta_tensor, (x_A, x_B), y, sigma=err, p0=[0.0])
    phi_hat = popt_b[0]
    chi2_b = weighted_chi2(y, model_b_xtheta_tensor(x_A, x_B, phi_hat), err)

    # Standard QM (no parameters)
    chi2_qm = weighted_chi2(y, qm_standard(x_A, x_B), err)

    return {
        "true_model": true_model_type,
        "true_param": true_param,
        "delta_hat": delta_hat,
        "phi_hat": phi_hat,
        "chi2_qm": chi2_qm,
        "chi2_a": chi2_a,
        "chi2_b": chi2_b,
        "delta_chi2_a": chi2_qm - chi2_a,
        "delta_chi2_b": chi2_qm - chi2_b,
    }

def wrapper_model_a(coords, delta): return model_a_detector_shift(coords[0], coords[1], delta)
def wrapper_model_b(coords, Phi): return model_b_xtheta_tensor(coords[0], coords[1], Phi)

## 4. Execution: Recovery Experiments

In [ ]:
params = [0.0, 0.01, 0.02, 0.05, 0.1]
settings = make_angle_scan_settings()
results = []

print("Running Model A (Detector Shift) recovery...")
for p in params:
    df = simulate_data(model_a_detector_shift, p, settings, n_trials_per_setting=50000)
    # Use wrapper for curve_fit compatibility
    x_A, x_B = df["theta_A"].values, df["theta_B"].values
    y, err = df["E"].values, df["Error"].values
    popt_a, _ = curve_fit(wrapper_model_a, (x_A, x_B), y, sigma=err, p0=[0.0])
    popt_b, _ = curve_fit(wrapper_model_b, (x_A, x_B), y, sigma=err, p0=[0.0])
    chi2_qm = weighted_chi2(y, qm_standard(x_A, x_B), err)
    chi2_a = weighted_chi2(y, wrapper_model_a((x_A, x_B), popt_a[0]), err)
    chi2_b = weighted_chi2(y, wrapper_model_b((x_A, x_B), popt_b[0]), err)
    results.append({"mode": "Model A", "true": p, "hat_a": popt_a[0], "hat_b": popt_b[0], "d_chi2_a": chi2_qm - chi2_a, "d_chi2_b": chi2_qm - chi2_b})

print("Running Model B (X-Theta Tensor) recovery...")
for p in params:
    df = simulate_data(model_b_xtheta_tensor, p, settings, n_trials_per_setting=50000)
    x_A, x_B = df["theta_A"].values, df["theta_B"].values
    y, err = df["E"].values, df["Error"].values
    popt_a, _ = curve_fit(wrapper_model_a, (x_A, x_B), y, sigma=err, p0=[0.0])
    popt_b, _ = curve_fit(wrapper_model_b, (x_A, x_B), y, sigma=err, p0=[0.0])
    chi2_qm = weighted_chi2(y, qm_standard(x_A, x_B), err)
    chi2_a = weighted_chi2(y, wrapper_model_a((x_A, x_B), popt_a[0]), err)
    chi2_b = weighted_chi2(y, wrapper_model_b((x_A, x_B), popt_b[0]), err)
    results.append({"mode": "Model B", "true": p, "hat_a": popt_a[0], "hat_b": popt_b[0], "d_chi2_a": chi2_qm - chi2_a, "d_chi2_b": chi2_qm - chi2_b})

df_results = pd.DataFrame(results)
display(df_results)

## 5. Visualizing Model Distinguishability

The residual signature is key. Model A ($δ$) produces a sine-wave residual: $ΔE ≈ δ \sin(θ_A - θ_B)$.
Model B ($Φ$) produces a cosine-wave residual: $ΔE ≈ (1-\cos(2Φ)) \cos(θ_A - θ_B)$.

In [ ]:
p_test = 0.1
df_a = simulate_data(model_a_detector_shift, p_test, settings)
df_b = simulate_data(model_b_xtheta_tensor, p_test, settings)

plt.figure(figsize=(10, 5))
plt.plot(df_a["angle_diff"], df_a["E"] - qm_standard(df_a["theta_A"], df_a["theta_B"]), 'o', label="Model A Residuals (Shift)")
plt.plot(df_b["angle_diff"], df_b["E"] - qm_standard(df_b["theta_A"], df_b["theta_B"]), 's', label="Model B Residuals (Tensor)")
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Angle Difference")
plt.ylabel("Residual ΔE (vs QM)")
plt.title("Residual Signature: Detector Shift vs Tensor Anisotropy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Conclusion

1. **Phase Recovery:** Direct curve fitting successfully recovers both phenomenological $\delta$ and physical $\Phi$.
2. **Model Selection:** $\Delta\chi^2$ clearly distinguishes both models from standard QM when the phase is non-zero.
3. **Symmetry:** The residual signature (sine vs cosine) allows for distinguishing between a simple detector shift and the physical tensor deformation model.

**Claim Classification:** This notebook is a **mathematical validation and methodology test**. It demonstrates that the X-Theta tensor model is distinguishable from both standard QM and simple detector-phase shifts, providing a clear path for future physical falsification.